# Session 47 Solutions: Saving and Loading Models

Data Science with Python, Level 3, Stage 7: Deployment

Try each exercise yourself before reading its solution. There is usually more than one correct answer; compare your approach with this one.

## Setup

In [1]:
import json
from datetime import date
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate, KFold

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
flats = pd.read_csv(DATA_URL + "apartments.csv")
X = flats.drop(columns=["apartment_id", "price_k"])
y = flats["price_k"]

## In-Class Exercises

### In-Class Exercise 1: Save the Churn Model

In [2]:
from sklearn.compose import make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
yc = (churn["churn"] == "Yes").astype(int)
Xc = churn.drop(columns=["customer_id", "churn"])
pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), make_column_selector(dtype_include="number")),
    ("cat", make_pipeline(SimpleImputer(strategy="constant", fill_value="None"), OneHotEncoder(handle_unknown="ignore")), make_column_selector(dtype_include=["object", "string"])),
    ("bool", "passthrough", make_column_selector(dtype_include="bool")),
])
churn_model = Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=1000))]).fit(Xc, yc)
Path("models").mkdir(exist_ok=True)
joblib.dump(churn_model, "models/churn_model.joblib")
again = joblib.load("models/churn_model.joblib")
print(np.allclose(again.predict_proba(Xc), churn_model.predict_proba(Xc)))

True


### In-Class Exercise 2: Read the Metadata

In [3]:
# Rebuild and save the lecture model, so this cell also works on its own
numeric = ["area_m2", "rooms", "floor", "building_age", "distance_center_km"]
prep = ColumnTransformer([("district", OneHotEncoder(handle_unknown="ignore"), ["district"]), ("numbers", StandardScaler(), numeric), ("flags", "passthrough", ["elevator", "parking"])])
Path("models").mkdir(exist_ok=True)
joblib.dump(Pipeline([("prep", prep), ("model", LinearRegression())]).fit(X, y), "models/price_model.joblib")
Path("models/metadata.json").write_text(json.dumps({"model_file": "price_model.joblib", "rows": len(X), "features": list(X.columns), "cv_mae_thousands": 10.1, "sklearn_version": sklearn.__version__}))
meta = json.loads(Path("models/metadata.json").read_text())
print(f"{meta['model_file']}: trained on {meta['rows']} rows, MAE {meta['cv_mae_thousands']}k, sklearn {meta['sklearn_version']}")

price_model.joblib: trained on 800 rows, MAE 10.1k, sklearn 1.9.1


### In-Class Exercise 3: Price Range

In [4]:
# Rebuild and save the lecture model, so this cell also works on its own
numeric = ["area_m2", "rooms", "floor", "building_age", "distance_center_km"]
prep = ColumnTransformer([("district", OneHotEncoder(handle_unknown="ignore"), ["district"]), ("numbers", StandardScaler(), numeric), ("flags", "passthrough", ["elevator", "parking"])])
Path("models").mkdir(exist_ok=True)
joblib.dump(Pipeline([("prep", prep), ("model", LinearRegression())]).fit(X, y), "models/price_model.joblib")
Path("models/metadata.json").write_text(json.dumps({"model_file": "price_model.joblib", "rows": len(X), "features": list(X.columns), "cv_mae_thousands": 10.1, "sklearn_version": sklearn.__version__}))
def price_range(price, mae):
    low = 5 * round((price - mae) / 5)
    high = 5 * round((price + mae) / 5)
    return f"between {low} and {high} thousand"

model = joblib.load("models/price_model.joblib")
row = pd.DataFrame([{"district": "Riverside", "area_m2": 85, "rooms": 3, "floor": 4, "building_age": 10, "distance_center_km": 3.5, "elevator": True, "parking": True}])
print(price_range(float(model.predict(row)[0]), 10))

between 110 and 130 thousand


### In-Class Exercise 4: Many Apartments at Once

In [5]:
# Rebuild and save the lecture model, so this cell also works on its own
numeric = ["area_m2", "rooms", "floor", "building_age", "distance_center_km"]
prep = ColumnTransformer([("district", OneHotEncoder(handle_unknown="ignore"), ["district"]), ("numbers", StandardScaler(), numeric), ("flags", "passthrough", ["elevator", "parking"])])
Path("models").mkdir(exist_ok=True)
joblib.dump(Pipeline([("prep", prep), ("model", LinearRegression())]).fit(X, y), "models/price_model.joblib")
Path("models/metadata.json").write_text(json.dumps({"model_file": "price_model.joblib", "rows": len(X), "features": list(X.columns), "cv_mae_thousands": 10.1, "sklearn_version": sklearn.__version__}))
model = joblib.load("models/price_model.joblib")
new = pd.DataFrame({
    "district": ["Old Town", "Hillview", "Westpark", "Northgate", "Riverside"],
    "area_m2": [60, 120, 75, 95, 140], "rooms": [2, 4, 3, 3, 5], "floor": [1, 6, 3, 8, 2],
    "building_age": [40, 5, 20, 12, 30], "distance_center_km": [0.8, 6.0, 4.5, 3.0, 7.5],
    "elevator": [False, True, True, True, False], "parking": [False, True, False, True, True],
})
new["predicted_price_k"] = model.predict(new).round(1)
new

,district,area_m2,rooms,floor,building_age,distance_center_km,elevator,parking,predicted_price_k
0,Old Town,60,2,1,40,0.8,False,False,79.0
1,Hillview,120,4,6,5,6.0,True,True,134.6
2,Westpark,75,3,3,20,4.5,True,False,58.1
3,Northgate,95,3,8,12,3.0,True,True,94.6
4,Riverside,140,5,2,30,7.5,False,True,147.0


## Part A: Basic

### Exercise 1: Save Anything

In [6]:
data = {"name": "test", "values": [1, 2, 3]}
joblib.dump(data, "demo.joblib")
print(joblib.load("demo.joblib") == data)

True


### Exercise 2: File Sizes

In [7]:
from sklearn.ensemble import RandomForestRegressor
num = ["area_m2", "rooms", "floor", "building_age", "distance_center_km"]
Path("models").mkdir(exist_ok=True)
for name, m in [("linear", LinearRegression()), ("forest", RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1))]:
    m.fit(X[num], y)
    path = Path(f"models/{name}.joblib")
    joblib.dump(m, path)
    print(name, round(path.stat().st_size / 1024, 1), "KB")
# A forest stores every split of every tree; a linear model stores six numbers.

linear 1.0 KB
forest 14233.5 KB


## Part B: Intermediate

### Exercise 3: Compressed Files

In [8]:
import time
from sklearn.ensemble import RandomForestRegressor
num = ["area_m2", "rooms", "floor", "building_age", "distance_center_km"]
forest = RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=-1).fit(X[num], y)
Path("models").mkdir(exist_ok=True)
for c in [0, 3]:
    path = Path(f"models/forest_c{c}.joblib")
    joblib.dump(forest, path, compress=c)
    t = time.perf_counter()
    joblib.load(path)
    print(c, round(path.stat().st_size / 1024), "KB", round(time.perf_counter() - t, 3), "s")

0 14233 KB 0.047 s
3 2949 KB 0.063 s


### Exercise 4: Validation Function

In [9]:
def validate(inputs, districts):
    p = []
    if not 20 <= inputs["area_m2"] <= 250:
        p.append("Area must be between 20 and 250.")
    if not 1 <= inputs["rooms"] <= 8:
        p.append("Rooms must be between 1 and 8.")
    elif inputs["rooms"] > inputs["area_m2"] / 10:
        p.append("Too many rooms for this area.")
    if not 0 <= inputs["floor"] <= 30:
        p.append("Floor must be between 0 and 30.")
    if not 0 <= inputs["building_age"] <= 120:
        p.append("Building age must be between 0 and 120.")
    if not 0 <= inputs["distance_center_km"] <= 30:
        p.append("Distance must be between 0 and 30 km.")
    if inputs["district"] not in districts:
        p.append("Unknown district.")
    for flag in ["elevator", "parking"]:
        if inputs[flag] not in (True, False):
            p.append(f"{flag} must be yes or no.")
    return p

districts = sorted(X["district"].unique())
good = {"area_m2": 85, "rooms": 3, "floor": 4, "building_age": 10, "distance_center_km": 3.5, "district": "Riverside", "elevator": True, "parking": True}
bad = {"area_m2": 40, "rooms": 6, "floor": 50, "building_age": 10, "distance_center_km": 3.5, "district": "Airport", "elevator": "maybe", "parking": True}
print(validate(good, districts))
print(validate(bad, districts))

[]
['Too many rooms for this area.', 'Floor must be between 0 and 30.', 'Unknown district.', 'elevator must be yes or no.']


## Part C: Challenge

### Exercise 5: A Model Class

Selecting `self.meta["features"]` puts the columns in the order used in training, whatever order the inputs arrive in.

In [10]:
# Rebuild and save the lecture model, so this cell also works on its own
numeric = ["area_m2", "rooms", "floor", "building_age", "distance_center_km"]
prep = ColumnTransformer([("district", OneHotEncoder(handle_unknown="ignore"), ["district"]), ("numbers", StandardScaler(), numeric), ("flags", "passthrough", ["elevator", "parking"])])
Path("models").mkdir(exist_ok=True)
joblib.dump(Pipeline([("prep", prep), ("model", LinearRegression())]).fit(X, y), "models/price_model.joblib")
Path("models/metadata.json").write_text(json.dumps({"model_file": "price_model.joblib", "rows": len(X), "features": list(X.columns), "cv_mae_thousands": 10.1, "sklearn_version": sklearn.__version__}))
class PricePredictor:
    def __init__(self, folder):
        folder = Path(folder)
        self.meta = json.loads((folder / "metadata.json").read_text())
        self.model = joblib.load(folder / self.meta["model_file"])

    def predict(self, **inputs):
        row = pd.DataFrame([inputs])[self.meta["features"]]
        return float(self.model.predict(row)[0])

    def describe(self):
        return f"Linear price model on {self.meta['rows']} flats, typical error {self.meta['cv_mae_thousands']} thousand"

pp = PricePredictor("models")
print(pp.describe())
print(round(pp.predict(district="Riverside", area_m2=85, rooms=3, floor=4, building_age=10, distance_center_km=3.5, elevator=True, parking=True), 1))

Linear price model on 800 flats, typical error 10.1 thousand
121.4


## Part D: Cumulative Project: Price Predictor App

### Exercise 6: Price Predictor App: Step 3, The Model Files

`%%writefile` is a Colab and Jupyter command that saves the cell's text to a file instead of running it. In Colab, download files from the folder panel on the left, or with `from google.colab import files; files.download("predict.py")`.

In [ ]:
%%writefile predict.py
import joblib
import pandas as pd

MODEL = joblib.load("models/price_model.joblib")

def predict_price(area_m2, rooms, floor, building_age, distance_center_km, district, elevator, parking):
    row = pd.DataFrame([{
        "district": district, "area_m2": area_m2, "rooms": rooms, "floor": floor,
        "building_age": building_age, "distance_center_km": distance_center_km,
        "elevator": elevator, "parking": parking,
    }])
    return float(MODEL.predict(row)[0])

## Part E: Quiz Answers

| Question | Answer | Explanation |
| --- | --- | --- |
| 1 | B | Train once, use many times. |
| 2 | B | The app sends raw inputs, so the preprocessing must come along. |
| 3 | B | Pin it in requirements.txt too. |
| 4 | B | Pickle-based files can execute code when loaded. |
| 5 | B | A model has no idea what is realistic. |